# 02 - Preprocessing

## Objetivo

Preparar las características para entrenar modelos de Machine Learning,
manteniendo un proceso reproducible y evitando data leakage.

En esta fase definiremos los conjuntos de entrenamiento y validación,
las estrategias para tratar valores ausentes y el pipeline de
preprocesamiento.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split


# Localizar los datos.
possible_paths = [Path("data"), Path("../data")]

DATA_DIR = next(
    path for path in possible_paths
    if (path / "train.csv").exists()
)

# Cargar los conjuntos originales.
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")


# Guardar los identificadores para la entrega final.
test_ids = test["Id"].copy()

# Separar características y variable objetivo.
X = train.drop(columns=["SalePrice", "Id"])
y = train["SalePrice"].copy()

# Eliminar el identificador de las características de test.
X_test = test.drop(columns="Id")

# Verificar que las columnas predictoras coinciden.
assert set(X.columns) == set(X_test.columns)

# Alinear las columnas de test con el orden de train.
X_test = X_test[X.columns]


# Crear conjuntos de entrenamiento y validación.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)


# Comprobar las dimensiones.
print("Entrenamiento:", X_train.shape, y_train.shape)
print("Validación:", X_valid.shape, y_valid.shape)
print("Test de Kaggle:", X_test.shape)

Entrenamiento: (1168, 79) (1168,)
Validación: (292, 79) (292,)
Test de Kaggle: (1459, 79)


In [15]:
import numpy as np
import pandas as pd


# Columnas categóricas donde NA significa que la característica
# no existe, según la descripción del dataset.
STRUCTURAL_NA_COLUMNS = [
    "Alley",
    "PoolQC",
    "MiscFeature",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
]


def apply_semantic_rules(df):
    """Aplica reglas basadas en el significado de las columnas."""
    df = df.copy()

    # 1. Ausencias categóricas que indican que la característica
    # no existe, según la documentación del dataset.
    for col in STRUCTURAL_NA_COLUMNS:
        df[col] = df[col].astype("object")
        df[col] = df[col].where(
            df[col].notna(),
            "None",
        )

    # 2. Distinguir ausencia de revestimiento y tipo desconocido.
    df["MasVnrType"] = df["MasVnrType"].astype("object")

    type_missing = df["MasVnrType"].isna()

    no_veneer = (
        type_missing
        & df["MasVnrArea"].eq(0)
    )

    unknown_type = (
        type_missing
        & ~df["MasVnrArea"].eq(0)
    )

    df.loc[no_veneer, "MasVnrType"] = "None"
    df.loc[unknown_type, "MasVnrType"] = "Unknown"

    # 3. Si no existe sótano, sus cantidades ausentes equivalen a cero.
    basement_numeric = [
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtUnfSF",
        "TotalBsmtSF",
        "BsmtFullBath",
        "BsmtHalfBath",
    ]

    no_basement = df["BsmtQual"].eq("None")

    for col in basement_numeric:
        df.loc[
            no_basement & df[col].isna(),
            col,
        ] = 0

    # 4. Si no existe garaje, capacidad y superficie ausentes equivalen
    # a cero. No hacemos esta sustitución en el año de construcción.
    no_garage = df["GarageType"].eq("None")

    for col in ["GarageCars", "GarageArea"]:
        df.loc[
            no_garage & df[col].isna(),
            col,
        ] = 0

    # 5. Tratar variables numéricas que representan categorías.
    for col in ["MSSubClass", "MoSold"]:
        df[col] = df[col].astype("string")

    return df

In [16]:
# Aplicar las reglas para identificar los tipos de variables.
X_train_rules = apply_semantic_rules(X_train)

numeric_features = (
    X_train_rules
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train_rules
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", len(categorical_features))

print("\n¿MSSubClass es categórica?",
      "MSSubClass" in categorical_features)

print("¿MoSold es categórica?",
      "MoSold" in categorical_features)

print("\nValores de MasVnrType tras aplicar las reglas:")
display(X_train_rules["MasVnrType"].value_counts(dropna=False))

print("\nValores ausentes restantes en columnas de ausencia estructural:")
display(X_train_rules[STRUCTURAL_NA_COLUMNS].isna().sum())

Variables numéricas: 34
Variables categóricas: 45

¿MSSubClass es categórica? True
¿MoSold es categórica? True

Valores de MasVnrType tras aplicar las reglas:


MasVnrType
None       672
BrkFace    366
Stone      106
BrkCmn      13
Unknown     11
Name: count, dtype: int64


Valores ausentes restantes en columnas de ausencia estructural:


Alley           0
PoolQC          0
MiscFeature     0
Fence           0
FireplaceQu     0
GarageType      0
GarageFinish    0
GarageQual      0
GarageCond      0
BsmtQual        0
BsmtCond        0
BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
dtype: int64

In [21]:
# Aplicar las reglas actualizadas.
X_train_clean = apply_semantic_rules(X_train)
X_valid_clean = apply_semantic_rules(X_valid)
X_test_clean = apply_semantic_rules(X_test)

# Volver a clasificar las características después de las reglas.
numeric_features = (
    X_train_clean
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train_clean
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", len(categorical_features))
print("Total de variables:", len(numeric_features) + len(categorical_features))

Variables numéricas: 34
Variables categóricas: 45
Total de variables: 79


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_preprocessor(scale_numeric=False):
    # Pipeline numérico.
    numeric_steps = [
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
    ]

    # El escalado será configurable según la familia de modelos.
    if scale_numeric:
        numeric_steps.append(
            ("scaler", StandardScaler())
        )

    numeric_transformer = Pipeline(
        steps=numeric_steps
    )

    # Pipeline categórico.
    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Missing",
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                ),
            ),
        ]
    )

    # Aplicar un tratamiento distinto a cada grupo de columnas.
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_transformer,
                numeric_features,
            ),
            (
                "cat",
                categorical_transformer,
                categorical_features,
            ),
        ],
        remainder="drop",
    )

    return preprocessor

In [23]:
# Crear y ajustar el preprocesador usando solo entrenamiento.
preprocessor = build_preprocessor(scale_numeric=False)

X_train_prepared = preprocessor.fit_transform(X_train_clean)
X_valid_prepared = preprocessor.transform(X_valid_clean)
X_test_prepared = preprocessor.transform(X_test_clean)

# Comprobar que los tres conjuntos son compatibles.
assert X_train_prepared.shape[1] == X_valid_prepared.shape[1]
assert X_train_prepared.shape[1] == X_test_prepared.shape[1]

print("Train:", X_train_prepared.shape)
print("Validación:", X_valid_prepared.shape)
print("Test:", X_test_prepared.shape)

# Mostrar valores numéricos que todavía necesitan imputación.
for name, df in [
    ("Train", X_train_clean),
    ("Validación", X_valid_clean),
    ("Test", X_test_clean),
]:
    missing = df[numeric_features].isna().sum()
    missing = missing[missing > 0]

    print(f"\nValores numéricos ausentes restantes en {name}:")
    display(missing)

Train: (1168, 330)
Validación: (292, 330)
Test: (1459, 330)

Valores numéricos ausentes restantes en Train:


LotFrontage    217
MasVnrArea       6
GarageYrBlt     64
dtype: int64


Valores numéricos ausentes restantes en Validación:


LotFrontage    42
MasVnrArea      2
GarageYrBlt    17
dtype: int64


Valores numéricos ausentes restantes en Test:


LotFrontage    227
MasVnrArea      15
GarageYrBlt     78
GarageCars       1
GarageArea       1
dtype: int64

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

def build_pipeline(estimator=None, scale_numeric=False):
    """
    Construye el pipeline de preprocessing y, opcionalmente,
    añade un modelo de Machine Learning.
    """
    steps = [
        (
            "semantic_rules",
            FunctionTransformer(
                apply_semantic_rules,
                validate=False,
            ),
        ),
        (
            "preprocessor",
            build_preprocessor(
                scale_numeric=scale_numeric
            ),
        ),
    ]

    if estimator is not None:
        steps.append(("model", estimator))

    return Pipeline(steps)